In [ ]:
path_prefix = '../..'

import sys
sys.path.append(path_prefix)

import json
import torch
from torch.utils.data import DataLoader

from egh_vlm.hallu_dataset import load_hallu_dataset, hallu_collate_fn, split_stratified
from egh_vlm.hallu_ffn_detector import HalluFFNDetector, eval_ffn_detector

## Analysis

In [ ]:
detector = HalluFFNDetector(2048, 2048, 1, 0.2)
detector.load_state_dict(torch.load(f'{path_prefix}/data/phd/prototype/detector_full.pt'))
detector.eval()

In [ ]:
with open(f'{path_prefix}/data/phd/prototype/phd_sample_qwen3_vl_2b.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)
features = load_hallu_dataset(f'{path_prefix}/data/phd/prototype/features_full.pt')
hidden_size = 2048

In [ ]:
train_dataset, val_dataset = split_stratified(
    features, train_ratio=0.7, random_state=42
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    collate_fn=hallu_collate_fn,
    shuffle=True
)
test_dataloader = DataLoader(
    val_dataset,
    batch_size=16,
    collate_fn=hallu_collate_fn,
    shuffle=True,
)

In [ ]:
result = eval_ffn_detector(detector, test_dataloader)
print(f'Evaluation: ACC={result["acc"]}, F1={result["f1"]}, PR AUC={result["pr_auc"]}')

## Dataset Construction

In [ ]:
scored_dataset = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_dataloader):
        ids, embs, grads, labels = batch
        scores = detector(embs, grads).squeeze()
        
        for id, score in zip(ids, scores):
            scored_dataset.append({
                'id': id,
                'detector_score': float(score),
            })

In [ ]:
test_dataset = []

for sample in scored_dataset:
    id = sample['id']
    data = next((s for s in dataset if s['id'] == id), None)
    if data is not None:
       data['detector_score'] = sample['detector_score']
       test_dataset.append(data)
    else:
        print(f'ID not found: {id}')
print(f'Length of test dataset: {len(test_dataset)}')

In [ ]:
save_path = f'{path_prefix}/data/phd/prototype/phd_sample_qwen3_vl_2b_scored.json'

with open(save_path, 'w') as f:
    json.dump(test_dataset, f, indent=4)